# Finetune SGMSE (sp-uhh/sgmse)
Notebook nay duoc viet lai tu file finetune FullSubNet cu, chuyen sang mo hinh **SGMSE** (score-based diffusion model cho speech enhancement): https://github.com/sp-uhh/sgmse

Diem khac biet quan trong so voi FullSubNet:
- SGMSE dung PyTorch Lightning, checkpoint la file `.ckpt` (khong phai `.tar`).
- Du lieu train/valid phai la cap file **da mix san** clean/noisy cung ten, nam trong `base_dir/train/clean`, `base_dir/train/noisy`, `base_dir/valid/clean`, `base_dir/valid/noisy` (khac FullSubNet - noi ban chi can list file clean.txt/noise.txt va mix on-the-fly).
- Notebook nay se: (1) mix du lieu clean+noise co san thanh cac cap train/valid cho SGMSE, (2) tai checkpoint pretrained tu Google Drive, (3) nap trong so pretrained vao model moi (finetune tu dau ve epoch/optimizer, giong cach lam voi FullSubNet), (4) train toi da 20 epoch voi **Early Stopping** de tu dong dung neu validation khong cai thien nua.

In [8]:
!nvidia-smi


Fri Sep 11 06:21:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Buoc 1: Clone repo va cai dat thu vien

In [9]:
%cd /kaggle/working
!rm -rf sgmse
!git clone https://github.com/sp-uhh/sgmse.git
%cd sgmse

!pip install -q -r requirements.txt
!pip install -q gdown soundfile librosa


/kaggle/working
Cloning into 'sgmse'...
remote: Enumerating objects: 1011, done.
remote: Counting objects: 100% (357/357), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 1011 (delta 276), reused 214 (delta 212), pack-reused 654 (from 3)
Receiving objects: 100% (1011/1011), 3.74 MiB | 13.92 MiB/s, done.
Resolving deltas: 100% (547/547), done.
/kaggle/working/sgmse


In [10]:
import torch
print("torch version:", torch.__version__)
print("torch cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Torch dang khong thay CUDA - restart session roi chay lai tu Buoc 0"


torch version: 2.10.0+cu128
torch cuda version: 12.8
cuda available: True


## Buoc 2: Duong dan du lieu (giu nguyen cau truc nhu file FullSubNet cu)

In [11]:
import os

CLEAN_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN"
NOISE_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE"
TEST_DIR  = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST "

WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.isdir(CLEAN_DIR), f"Khong tim thay: {CLEAN_DIR}"
assert os.path.isdir(NOISE_DIR), f"Khong tim thay: {NOISE_DIR}"
assert os.path.isdir(TEST_DIR), f"Khong tim thay: {TEST_DIR}"
print("OK, cac thu muc du lieu ton tai.")


OK, cac thu muc du lieu ton tai.


In [12]:
AUDIO_EXT = (".wav", ".flac")

def list_audio_files(folder, exts=AUDIO_EXT):
    files = []
    for root, _, names in os.walk(folder):
        for n in names:
            if n.lower().endswith(exts):
                files.append(os.path.join(root, n))
    return sorted(files)

clean_files = list_audio_files(CLEAN_DIR)
noise_files = list_audio_files(NOISE_DIR)
test_files  = list_audio_files(TEST_DIR)

print("So file clean:", len(clean_files))
print("So file noise:", len(noise_files))
print("So file test (noisy):", len(test_files))

assert len(clean_files) > 0, "Khong tim thay file audio nao trong CLEAN_DIR"
assert len(noise_files) > 0, "Khong tim thay file audio nao trong NOISE_DIR"


So file clean: 8460
So file noise: 8460
So file test (noisy): 940


## Buoc 3: Ghep cap du lieu clean/noisy cho SGMSE - dung truc tiep tu /kaggle/input, KHONG copy/resample
**Luu y quan trong:** `NOISE_DIR` khong phai noise thuan ma la du lieu **da mix san** (noisy speech) song song voi `CLEAN_DIR`.

SGMSE can cau truc thu muc `base_dir/train/clean`, `base_dir/train/noisy`, `base_dir/valid/clean`, `base_dir/valid/noisy`, voi file clean/noisy **cung ten** trong moi cap. Vi `/kaggle/input` la read-only va ban muon dung thang du lieu goc (khong tao ban sao / khong resample duoi `outputs`), cell duoi day chi **tao symlink** tro thang toi file goc trong `CLEAN_DIR`/`NOISE_DIR` - khong doc/ghi lai noi dung audio. Chi khi file KHONG phai dinh dang `.wav` (SGMSE chi doc `.wav`) thi moi bat buoc phai convert (ghi file moi).

In [13]:
import random, os, soundfile as sf

random.seed(0)

def basename_no_ext(path):
    return os.path.splitext(os.path.basename(path))[0]

clean_by_key = {basename_no_ext(f): f for f in clean_files}
noisy_by_key = {basename_no_ext(f): f for f in noise_files}   # NOISE_DIR = noisy speech da mix san

common_keys = sorted(set(clean_by_key) & set(noisy_by_key))
only_clean = set(clean_by_key) - set(noisy_by_key)
only_noisy = set(noisy_by_key) - set(clean_by_key)

print("So cap clean/noisy ghep duoc theo ten file:", len(common_keys))
print("So file clean khong co cap noisy tuong ung:", len(only_clean))
print("So file noisy khong co cap clean tuong ung:", len(only_noisy))

assert len(common_keys) > 0, (
    "Khong ghep duoc cap nao theo ten file trung khop. Kiem tra lai quy uoc dat ten file giua "
    "CLEAN_DIR va NOISE_DIR (co the can ghep theo prefix/suffix khac thay vi trung ten hoan toan)."
)

# Kiem tra nhanh sample rate cua vai file (chi doc header, khong load full audio) de canh bao
# neu du lieu khong dong nhat 16kHz - SGMSE pretrained thuong dung 16kHz.
sample_keys = random.sample(common_keys, min(5, len(common_keys)))
for k in sample_keys:
    sr_c = sf.info(clean_by_key[k]).samplerate
    sr_n = sf.info(noisy_by_key[k]).samplerate
    print(f"  {k}: clean sr={sr_c}, noisy sr={sr_n}")


So cap clean/noisy ghep duoc theo ten file: 8460
So file clean khong co cap noisy tuong ung: 0
So file noisy khong co cap clean tuong ung: 0
  RLMBV011-N14: clean sr=16000, noisy sr=16000
  RLMNV005-N29: clean sr=16000, noisy sr=16000
  AIBDV017-N06: clean sr=16000, noisy sr=16000
  RLBDV003-N13: clean sr=16000, noisy sr=16000
  RLQNV019-N11: clean sr=16000, noisy sr=16000


In [14]:
BASE_DIR = os.path.join(WORK_DIR, "sgmse_data")
TRAIN_CLEAN_DIR = os.path.join(BASE_DIR, "train", "clean")
TRAIN_NOISY_DIR = os.path.join(BASE_DIR, "train", "noisy")
VALID_CLEAN_DIR = os.path.join(BASE_DIR, "valid", "clean")
VALID_NOISY_DIR = os.path.join(BASE_DIR, "valid", "noisy")
for d in [TRAIN_CLEAN_DIR, TRAIN_NOISY_DIR, VALID_CLEAN_DIR, VALID_NOISY_DIR]:
    os.makedirs(d, exist_ok=True)

N_VAL = min(40, len(common_keys) // 20 + 1)
val_keys = set(random.sample(common_keys, min(N_VAL, len(common_keys))))
train_keys = [k for k in common_keys if k not in val_keys]

print("So cap dung cho validation:", len(val_keys))
print("So cap dung cho train:", len(train_keys))

def link_or_convert(src_path, dst_dir, key):
    """Tao symlink toi file goc (khong copy/resample). Chi convert (ghi file moi) neu
    duoi file khong phai .wav, vi SGMSE chi doc duoc .wav."""
    ext = os.path.splitext(src_path)[1].lower()
    dst_path = os.path.join(dst_dir, f"{key}.wav")
    if os.path.lexists(dst_path):
        return
    if ext == ".wav":
        os.symlink(os.path.abspath(src_path), dst_path)
    else:
        wav, sr = sf.read(src_path, always_2d=False)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        sf.write(dst_path, wav, sr)

def link_pairs(keys, out_clean_dir, out_noisy_dir):
    for k in keys:
        link_or_convert(clean_by_key[k], out_clean_dir, k)
        link_or_convert(noisy_by_key[k], out_noisy_dir, k)

link_pairs(val_keys, VALID_CLEAN_DIR, VALID_NOISY_DIR)
link_pairs(train_keys, TRAIN_CLEAN_DIR, TRAIN_NOISY_DIR)

print("Da tao xong cau truc thu muc (symlink toi du lieu goc, khong copy) tai:", BASE_DIR)


So cap dung cho validation: 40
So cap dung cho train: 8420
Da tao xong cau truc thu muc (symlink toi du lieu goc, khong copy) tai: /kaggle/working/sgmse_data


## Buoc 4: Tai checkpoint pretrained tu Google Drive
Link ban dua: https://drive.google.com/file/d/1-hYFg7bHZWImf5U-9EY3KVuzKL0GsCva/view?usp=drive_link

In [20]:
import gdown

DRIVE_FILE_ID = "19DALghJmDbXI0Dve5jkI_cW383aoXM2m"
PRETRAINED_CKPT = "/kaggle/working/pretrained_sgmse.ckpt"

gdown.download(id=DRIVE_FILE_ID, output=PRETRAINED_CKPT, quiet=False, fuzzy=True)

size_mb = os.path.getsize(PRETRAINED_CKPT) / (1024 * 1024)
print(f"Da tai ve: {PRETRAINED_CKPT} ({size_mb:.2f} MB)")

with open(PRETRAINED_CKPT, "rb") as f:
    head = f.read(200)

is_html = head.lstrip()[:20].lower().startswith((b"<!doctype", b"<html"))
is_zip_or_pickle = head[:2] == b"PK" or head[:1] == b"\x80"

if is_html or size_mb < 100 or not is_zip_or_pickle:
    print("\n--- 200 byte dau tien cua file tai ve (de debug) ---")
    print(head)
    msg = f"""File tai ve co ve KHONG phai checkpoint SGMSE day du (~1.3GB).
Dung luong thuc te: {size_mb:.2f} MB.
Cach khac phuc:
  1) Mo trinh duyet, vao link Drive, bam 'Download anyway' 1 lan de xac nhan, roi thu lai.
  2) Hoac chay: !gdown --id {DRIVE_FILE_ID} -O {PRETRAINED_CKPT} --fuzzy
  3) Hoac upload checkpoint len Kaggle Dataset roi Add Input, tro thang PRETRAINED_CKPT toi do."""
    raise RuntimeError(msg)

print("File co ve la checkpoint hop le, dung luong hop ly, tiep tuc.")

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=19DALghJmDbXI0Dve5jkI_cW383aoXM2m

but Gdown can't. Please check connections and permissions.

In [16]:
# Kiem tra cac hyperparameter da luu trong checkpoint (backbone, sde, lr, ...)
# de biet chinh xac kien truc goc, tranh mismatch khi nap lai model.
raw_ckpt = torch.load(PRETRAINED_CKPT, map_location="cpu", weights_only=False)
print("Cac key chinh trong checkpoint:", list(raw_ckpt.keys()))
hparams = raw_ckpt.get("hyper_parameters", {})
print("\nhyper_parameters da luu trong checkpoint:")
for k, v in hparams.items():
    print(f"  {k}: {v}")


RuntimeError: [enforce fail at inline_container.cc:180] . file in archive is not in a subdirectory: custom_checkpoint_0.pkl

## Buoc 5: Kiem tra chinh xac ten tham so cua SGMSE (tu dong thich nghi theo phien ban repo)
Vi API cua sgmse co the thay doi theo phien ban, chay 2 cell duoi de xem chinh xac cac tham so cua `SpecsDataModule` (vd `base_dir`, `batch_size`, `num_workers`) va cua `ScoreModel` (vd `lr`, `num_eval_files`, `t_eps`...) truoc khi nap checkpoint o Buoc 6. Neu ten tham so o Buoc 6 bi loi (TypeError: unexpected keyword), doi lai theo dung ten in ra o day.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/sgmse")

import argparse
from sgmse.data_module import SpecsDataModule
from sgmse.model import ScoreModel

tmp_parser = argparse.ArgumentParser()
SpecsDataModule.add_argparse_args(tmp_parser.add_argument_group("DataModule"))
print("=== Tham so cua SpecsDataModule (dataset/dataloader) ===")
tmp_parser.print_help()


In [ ]:
tmp_parser2 = argparse.ArgumentParser()
ScoreModel.add_argparse_args(tmp_parser2.add_argument_group("ScoreModel"))
print("=== Tham so cua ScoreModel (lr, num_eval_files, loss_type, ...) ===")
tmp_parser2.print_help()


## Buoc 6: Nap trong so pretrained vao model moi (finetune tu epoch 0)
Giong cach lam voi FullSubNet (nap state_dict vao model sach roi luu lai), o day ta dung `ScoreModel.load_from_checkpoint(...)` cua Lightning: no tu dong doc lai kien truc (backbone/sde) da luu trong checkpoint, chi ghi de cac tham so du lieu (`base_dir`, `batch_size`, `num_workers`, `num_eval_files`) de tro toi du lieu moi cua ban. Optimizer va epoch se **bat dau lai tu dau** (khong resume trainer state cu) - dung nghia la finetune.

In [ ]:
BATCH_SIZE = 8        # giam neu bi OOM tren GPU Kaggle
NUM_WORKERS = 2
NUM_EVAL_FILES = min(10, len(val_keys))   # so file dung de tinh PESQ/SI-SDR moi epoch

model = ScoreModel.load_from_checkpoint(
    PRETRAINED_CKPT,
    base_dir=BASE_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    num_eval_files=NUM_EVAL_FILES,
)
print("Da nap xong model tu checkpoint pretrained, san sang finetune tren du lieu moi.")


## Buoc 7: Cau hinh Trainer voi Early Stopping (toi da 20 epoch)
Mac dinh theo doi metric `pesq` (cang cao cang tot). Neu checkpoint cua ban duoc nap voi `num_eval_files=0` (khong tinh PESQ), doi `EARLY_STOP_MONITOR` sang `"valid_loss"` va `EARLY_STOP_MODE` sang `"min"`.

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks.early_stopping import EarlyStopping

MAX_EPOCHS = 20
EARLY_STOP_MONITOR = "pesq"     # doi thanh "valid_loss" neu NUM_EVAL_FILES = 0
EARLY_STOP_MODE = "max"         # doi thanh "min" neu dung valid_loss
EARLY_STOP_PATIENCE = 5          # so epoch lien tiep khong cai thien truoc khi dung som

SAVE_DIR = os.path.join(WORK_DIR, "sgmse_experiments")
os.makedirs(SAVE_DIR, exist_ok=True)

early_stop_cb = EarlyStopping(
    monitor=EARLY_STOP_MONITOR,
    mode=EARLY_STOP_MODE,
    patience=EARLY_STOP_PATIENCE,
    verbose=True,
)

ckpt_best_cb = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="best-{epoch}-{" + EARLY_STOP_MONITOR + ":.3f}",
    monitor=EARLY_STOP_MONITOR,
    mode=EARLY_STOP_MODE,
    save_top_k=1,
)

ckpt_last_cb = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="last-{epoch}",
    save_last=True,
)

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=MAX_EPOCHS,
    callbacks=[early_stop_cb, ckpt_best_cb, ckpt_last_cb],
    logger=pl.loggers.CSVLogger(save_dir=SAVE_DIR, name="finetune_logs"),
    log_every_n_steps=10,
    num_sanity_val_steps=0,
    accumulate_grad_batches=1,
)

print(f"Trainer san sang: toi da {MAX_EPOCHS} epoch, early stopping theo '{EARLY_STOP_MONITOR}' "
      f"(mode={EARLY_STOP_MODE}, patience={EARLY_STOP_PATIENCE})")


## Buoc 8: Chay finetune

In [ ]:
trainer.fit(model)
print("Hoan tat finetune (hoac da dung som do early stopping).")
print("So epoch thuc te da chay:", trainer.current_epoch)


In [ ]:
print("Checkpoint tot nhat (theo", EARLY_STOP_MONITOR, "):", ckpt_best_cb.best_model_path)
print("Checkpoint cuoi cung:", ckpt_last_cb.last_model_path)


## (Tuy chon) Ve bieu do metric qua cac epoch de xem early stopping hoat dong the nao

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob

csv_logs = sorted(glob.glob(os.path.join(SAVE_DIR, "finetune_logs", "version_*", "metrics.csv")))
if csv_logs:
    df = pd.read_csv(csv_logs[-1])
    cols = [c for c in df.columns if c in ("pesq", "si_sdr", "valid_loss", "train_loss")]
    if cols:
        df_epoch = df.groupby("epoch")[cols].mean(numeric_only=True)
        df_epoch.plot(marker="o", figsize=(8, 5), title="Metrics theo epoch")
        plt.xlabel("epoch")
        plt.grid(True)
        plt.show()
    else:
        print("Khong tim thay cot metric quen thuoc trong log, cot hien co:", list(df.columns))
else:
    print("Chua tim thay file metrics.csv, kiem tra lai SAVE_DIR.")


## Buoc 9: Kiem tra tham so cua enhancement.py truoc khi chay inference tren TEST_DIR

In [ ]:
%cd /kaggle/working/sgmse
!python enhancement.py --help


In [ ]:
BEST_CKPT = ckpt_best_cb.best_model_path or ckpt_last_cb.last_model_path
OUTPUT_DIR = "/kaggle/working/enhanced_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Se dung checkpoint:", BEST_CKPT)

%cd /kaggle/working/sgmse
!python enhancement.py --test_dir "{TEST_DIR}" --enhanced_dir "{OUTPUT_DIR}" --ckpt "{BEST_CKPT}"
